# 🏪 Contoso Sales Analytics — Sample Data Setup

This notebook creates sample data for the **Contoso Sales Analytics** demo.  
It generates two tables in the attached Lakehouse:

| Table | Description |
|-------|-------------|
| **Products** | 50 products across 8 categories with realistic names and prices |
| **SalesTransactions** | 500 sales records with dates, regions, reps, and computed totals |

**How to use:** Attach a Lakehouse to this notebook, then click **Run All**.

## Step 1: Configuration

Adjust the parameters below to control how much sample data is generated.  
The table names determine what the Delta tables will be called in your Lakehouse.

In [ ]:
# Configuration — adjust if needed
NUM_PRODUCTS = 50
NUM_TRANSACTIONS = 500
PRODUCTS_TABLE = "products"
SALES_TABLE = "sales_transactions"

## Step 2: Generate Products Data

In [ ]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DecimalType
from decimal import Decimal

products_data = [
    (1,  "Wireless Headphones",       "Electronics",       Decimal("79.99")),
    (2,  "4K Ultra HD Monitor",       "Electronics",       Decimal("449.99")),
    (3,  "Bluetooth Speaker",         "Electronics",       Decimal("59.99")),
    (4,  "Laptop Stand",              "Electronics",       Decimal("34.99")),
    (5,  "Mechanical Keyboard",       "Electronics",       Decimal("129.99")),
    (6,  "USB-C Hub",                 "Electronics",       Decimal("49.99")),
    (7,  "Noise Cancelling Earbuds",  "Electronics",       Decimal("199.99")),
    (8,  "Running Shoes",             "Clothing",          Decimal("89.99")),
    (9,  "Denim Jacket",              "Clothing",          Decimal("64.99")),
    (10, "Cotton T-Shirt",            "Clothing",          Decimal("19.99")),
    (11, "Winter Parka",              "Clothing",          Decimal("149.99")),
    (12, "Yoga Pants",                "Clothing",          Decimal("44.99")),
    (13, "Wool Sweater",              "Clothing",          Decimal("59.99")),
    (14, "Garden Hose (50ft)",        "Home & Garden",     Decimal("29.99")),
    (15, "Stainless Steel Cookware Set", "Home & Garden", Decimal("129.99")),
    (16, "LED Desk Lamp",             "Home & Garden",     Decimal("39.99")),
    (17, "Scented Candle Set",        "Home & Garden",     Decimal("24.99")),
    (18, "Robot Vacuum",              "Home & Garden",     Decimal("299.99")),
    (19, "Throw Blanket",             "Home & Garden",     Decimal("34.99")),
    (20, "Mountain Bike",             "Sports",            Decimal("549.99")),
    (21, "Yoga Mat",                  "Sports",            Decimal("29.99")),
    (22, "Dumbbell Set",              "Sports",            Decimal("89.99")),
    (23, "Tennis Racket",             "Sports",            Decimal("79.99")),
    (24, "Camping Tent (4-Person)",   "Sports",            Decimal("199.99")),
    (25, "Basketball",                "Sports",            Decimal("29.99")),
    (26, "Organic Coffee Beans",      "Food & Beverages",  Decimal("14.99")),
    (27, "Green Tea Collection",      "Food & Beverages",  Decimal("9.99")),
    (28, "Dark Chocolate Assortment", "Food & Beverages",  Decimal("19.99")),
    (29, "Extra Virgin Olive Oil",    "Food & Beverages",  Decimal("12.99")),
    (30, "Protein Bar Variety Pack",  "Food & Beverages",  Decimal("24.99")),
    (31, "Sparkling Water (12-Pack)", "Food & Beverages",  Decimal("8.99")),
    (32, "Science Fiction Anthology", "Books",             Decimal("16.99")),
    (33, "Data Science Handbook",     "Books",             Decimal("44.99")),
    (34, "Cookbook: World Cuisines",   "Books",             Decimal("29.99")),
    (35, "Mystery Novel Collection",  "Books",             Decimal("22.99")),
    (36, "Business Strategy Guide",   "Books",             Decimal("34.99")),
    (37, "Children's Picture Book",   "Books",             Decimal("12.99")),
    (38, "Vitamin C Supplements",     "Health & Beauty",   Decimal("15.99")),
    (39, "Sunscreen SPF 50",          "Health & Beauty",   Decimal("11.99")),
    (40, "Electric Toothbrush",       "Health & Beauty",   Decimal("69.99")),
    (41, "Moisturizing Face Cream",   "Health & Beauty",   Decimal("24.99")),
    (42, "Essential Oils Kit",        "Health & Beauty",   Decimal("34.99")),
    (43, "Hair Dryer Pro",            "Health & Beauty",   Decimal("54.99")),
    (44, "Building Blocks Set",       "Toys",              Decimal("39.99")),
    (45, "Remote Control Car",        "Toys",              Decimal("49.99")),
    (46, "Board Game: Strategy Quest","Toys",              Decimal("29.99")),
    (47, "Stuffed Animal Bear",       "Toys",              Decimal("19.99")),
    (48, "Art Supplies Kit",          "Toys",              Decimal("34.99")),
    (49, "Puzzle 1000 Pieces",        "Toys",              Decimal("17.99")),
    (50, "Smartwatch Pro",            "Electronics",       Decimal("299.99"))
]

products_schema = StructType([
    StructField("ProductID", IntegerType(), False),
    StructField("ProductName", StringType(), False),
    StructField("Category", StringType(), False),
    StructField("UnitPrice", DecimalType(10, 2), False)
])

products_df = spark.createDataFrame(products_data[:NUM_PRODUCTS], schema=products_schema)

products_df.show(10, truncate=False)
print(f"✅ Generated {products_df.count()} products")

## Step 3: Generate Sales Transactions

In [ ]:
import random
from datetime import date, timedelta
from pyspark.sql import functions as F
from pyspark.sql.types import DateType

rng = random.Random(42)

sales_reps = [
    "Alice Johnson", "Bob Smith", "Carlos Rivera", "Diana Chen",
    "Ethan Brooks", "Fiona O'Brien", "George Kim", "Hannah Patel",
    "Ivan Petrov", "Julia Santos"
]

regions = [
    "North America", "Europe", "Asia Pacific",
    "Latin America", "Middle East & Africa"
]

start_date = date(2024, 1, 1)
end_date = date(2025, 12, 31)
date_range_days = (end_date - start_date).days

# Build a price lookup from the products list
price_lookup = {pid: price for pid, _, _, price in products_data[:NUM_PRODUCTS]}

transactions = []
for i in range(1, NUM_TRANSACTIONS + 1):
    product_id = rng.randint(1, NUM_PRODUCTS)
    quantity = rng.randint(1, 20)
    unit_price = price_lookup[product_id]
    total_amount = Decimal(str(quantity)) * unit_price
    tx_date = start_date + timedelta(days=rng.randint(0, date_range_days))
    transactions.append((
        i,
        product_id,
        rng.choice(sales_reps),
        rng.choice(regions),
        quantity,
        total_amount,
        tx_date
    ))

sales_schema = StructType([
    StructField("TransactionID", IntegerType(), False),
    StructField("ProductID", IntegerType(), False),
    StructField("SalesRep", StringType(), False),
    StructField("Region", StringType(), False),
    StructField("Quantity", IntegerType(), False),
    StructField("TotalAmount", DecimalType(10, 2), False),
    StructField("TransactionDate", DateType(), False)
])

sales_df = spark.createDataFrame(transactions, schema=sales_schema)

sales_df.show(10, truncate=False)
print(f"✅ Generated {sales_df.count()} sales transactions")

## Step 4: Write Delta Tables to Lakehouse

In [ ]:
products_df.write.mode("overwrite").format("delta").saveAsTable(PRODUCTS_TABLE)
print(f"✅ Wrote {products_df.count()} rows to '{PRODUCTS_TABLE}'")

sales_df.write.mode("overwrite").format("delta").saveAsTable(SALES_TABLE)
print(f"✅ Wrote {sales_df.count()} rows to '{SALES_TABLE}'")

## Step 5: Verify

In [ ]:
print("📊 Lakehouse Tables Summary")
print("=" * 40)
for table_name in [PRODUCTS_TABLE, SALES_TABLE]:
    df = spark.sql(f"SELECT * FROM {table_name}")
    print(f"\n🔹 {table_name}: {df.count()} rows, {len(df.columns)} columns")
    df.show(5, truncate=False)

## ✅ Done!

Your Lakehouse now contains the sample data. Next steps:

1. Go back to your workspace
2. Create a **Data Agent** and connect it to this Lakehouse
3. See [02-create-data-agent.md](../docs/02-create-data-agent.md) for detailed instructions